# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ma5029blp-wq/ML-flyrank-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
df = pd.read_csv("content_refresh_anonymized.csv")
df["declining_observed"] = (df["trend_direction"] == "down").astype(int)
numeric_features = [
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "search_volume",
    "competition",
    "word_count",
    "char_count"
]

categorical_features = [
    "content_type",
    "main_intent"
]

features = numeric_features + categorical_features

X = df[features]
y = df["declining_observed"]
groups = df["client_id"]
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])
model.fit(X_train, y_train)
test_probabilities = model.predict_proba(X_test)[:, 1]

print("Setup complete")
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", df.iloc[train_idx]["client_id"].nunique())
print("Test clients:", df.iloc[test_idx]["client_id"].nunique())
print("Client overlap:",
      len(set(df.iloc[train_idx]["client_id"]) &
          set(df.iloc[test_idx]["client_id"])))
print("Test base rate:", y_test.mean())

Setup complete
Training rows: 22885
Test rows: 7115
Training clients: 24
Test clients: 8
Client overlap: 0
Test base rate: 0.516514406184118


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

queue = X_test.copy()

queue["decline_score"] = test_probabilities
queue["actual_declining"] = y_test.values
queue["reason_code"] = np.where(
    (queue["days_since_last_update"] >= 91) &
    (queue["impressions_prev_30d"] > 0),
    "VISIBLE_STALE",
    "HIGH_MODEL_SCORE"
)

queue["recommended_action"] = "REVIEW_REFRESH"

queue = queue.sort_values(
    "decline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = queue.index + 1
queue[
    [
        "rank",
        "decline_score",
        "recommended_action",
        "reason_code",
        "days_since_last_update",
        "impressions_prev_30d"
    ]
].head(20)

,rank,decline_score,recommended_action,reason_code,days_since_last_update,impressions_prev_30d
0,1,0.864996,REVIEW_REFRESH,HIGH_MODEL_SCORE,20,84773
1,2,0.839192,REVIEW_REFRESH,HIGH_MODEL_SCORE,20,64917
2,3,0.838248,REVIEW_REFRESH,HIGH_MODEL_SCORE,20,97200
3,4,0.806764,REVIEW_REFRESH,HIGH_MODEL_SCORE,20,97606
4,5,0.803101,REVIEW_REFRESH,VISIBLE_STALE,106,30774
5,6,0.798786,REVIEW_REFRESH,HIGH_MODEL_SCORE,20,54506
6,7,0.777023,REVIEW_REFRESH,HIGH_MODEL_SCORE,20,84550
7,8,0.775210,REVIEW_REFRESH,VISIBLE_STALE,106,9919
8,9,0.775169,REVIEW_REFRESH,HIGH_MODEL_SCORE,89,21552
9,10,0.774384,REVIEW_REFRESH,VISIBLE_STALE,104,218786


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended to help an SEO or content reviewer decide which pages to review first. The Logistic Regression score provides a ranking signal, while the reason code gives some context for the recommendation.

The output is decision-support, not an automatic publishing or editing system. A high score means the page was ranked highly for observed decline in the tested data; it does not mean that the page must be refreshed.

The results were validated on a grouped client-level test split, so the test clients were not seen during training. The model was evaluated on this portfolio and should not be assumed to perform the same way for every future client or time period.

The playbook also does not establish that refreshing a page will cause its performance to improve. Content quality, search intent, competition, and other factors still need human review.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Basic checks supporting the intended-use description

print("=== Intended Use Checks ===")
print("Test rows:", len(queue))
print("Test base rate:", round(y_test.mean(), 4))
print("Top score:", round(queue["decline_score"].max(), 4))
print("Lowest score:", round(queue["decline_score"].min(), 4))

print("\nRecommended actions:")
print(queue["recommended_action"].value_counts())

print("\nReason codes:")
print(queue["reason_code"].value_counts())

=== Intended Use Checks ===
Test rows: 7115
Test base rate: 0.5165
Top score: 0.865
Lowest score: 0.0016

Recommended actions:
recommended_action
REVIEW_REFRESH    7115
Name: count, dtype: int64

Reason codes:
reason_code
HIGH_MODEL_SCORE    5993
VISIBLE_STALE       1122
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and no-go list

A person must review each recommended page before taking action. The reviewer should check whether the page still matches the search intent, whether the information is accurate and up to date, whether important sections are missing, and whether the page still has useful search visibility.

The model score alone should not decide whether content is rewritten, removed, redirected, or published. A high score is a reason to review the page, not a final decision.

The following should not be automated:

- Publishing or rewriting content without human approval.
- Deleting or redirecting a page based only on the model score.
- Treating a high score as proof that a refresh will improve performance.
- Changing search intent or keyword targeting automatically.
- Making claims about Google rankings or future traffic from the score alone.
- Applying recommendations when the underlying data is missing, clearly unusual, or outside the validated population.

Human review is especially important for high-value pages, sensitive topics, major business pages, and cases where the recommendation conflicts with editorial or SEO context.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Simple checks for human-review requirements

print("=== Human Review Checks ===")

print("Total ranked items:", len(queue))

print("\nItems requiring human review:")
print(queue["recommended_action"].value_counts())

print("\nHighest-priority items:")
print(
    queue[
        [
            "rank",
            "decline_score",
            "reason_code",
            "days_since_last_update",
            "impressions_prev_30d"
        ]
    ].head(10)
)

print("\nNo automatic publishing, deletion, redirect, or rewrite actions are included.")
print("Final action remains with a human reviewer.")

=== Human Review Checks ===
Total ranked items: 7115

Items requiring human review:
recommended_action
REVIEW_REFRESH    7115
Name: count, dtype: int64

Highest-priority items:
   rank  decline_score       reason_code  days_since_last_update  \
0     1       0.864996  HIGH_MODEL_SCORE                      20   
1     2       0.839192  HIGH_MODEL_SCORE                      20   
2     3       0.838248  HIGH_MODEL_SCORE                      20   
3     4       0.806764  HIGH_MODEL_SCORE                      20   
4     5       0.803101     VISIBLE_STALE                     106   
5     6       0.798786  HIGH_MODEL_SCORE                      20   
6     7       0.777023  HIGH_MODEL_SCORE                      20   
7     8       0.775210     VISIBLE_STALE                     106   
8     9       0.775169  HIGH_MODEL_SCORE                      89   
9    10       0.774384     VISIBLE_STALE                     104   

   impressions_prev_30d  
0                 84773  
1                 6491

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The playbook should be reviewed periodically because search performance, content mix, and client populations can change over time.

The main monitoring checks are:

- **Ranking performance:** re-check Precision@20 and Precision@50 on a new labeled period.
- **Base rate:** monitor whether the observed decline rate changes substantially from the validation period.
- **Data quality:** check for missing or unusual values in important input features.
- **Population changes:** check whether new clients or content types differ from the population used for validation.
- **Model drift:** compare the distribution of model scores with the validation period.

A model review or retraining should be considered if Precision@20 or Precision@50 drops meaningfully on a new labeled period, the decline base rate changes substantially, important feature distributions shift, or new content/client populations are materially different from the validation data.

These are review triggers rather than fixed production thresholds because this notebook is a research and decision-support workflow, not a production monitoring system.

In [11]:
# Monitoring snapshot for the current validation set

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()

current_p20 = precision_at_k(test_probabilities, y_test, 20)
current_p50 = precision_at_k(test_probabilities, y_test, 50)

print("=== Monitoring Snapshot ===")

print("Current test base rate:",
      round(y_test.mean(), 4))

print("Current Precision@20:",
      round(current_p20, 4))

print("Current Precision@50:",
      round(current_p50, 4))

print("\nModel score distribution:")
print(
    queue["decline_score"].describe()[
        ["min", "25%", "50%", "75%", "max"]
    ]
)

print("\nMissing values in model features:")
print(
    X_test[features]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("\nMonitoring note:")
print("Re-check the model on a new labeled period before relying on the queue again.")

=== Monitoring Snapshot ===
Current test base rate: 0.5165
Current Precision@20: 0.5
Current Precision@50: 0.62

Model score distribution:
min    0.001578
25%    0.443079
50%    0.548607
75%    0.644861
max    0.864996
Name: decline_score, dtype: float64

Missing values in model features:
word_count                1085
char_count                1085
search_volume              149
competition                149
main_intent                 18
sessions_prev_30d            0
clicks_prev_30d              0
impressions_prev_30d         0
days_since_last_update       0
avg_position                 0
dtype: int64

Monitoring note:
Re-check the model on a new labeled period before relying on the queue again.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

The ranked review queue is exported so the paper can reuse the same output produced by this notebook. The export contains the model ranking, score, recommended review action, reason code, and supporting content signals.

The queue is an analysis artifact rather than a production dataset. It should be regenerated by running this notebook rather than treated as a permanent prediction file.

The exported queue is saved under `work/outputs/` as required by the assignment.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Make sure the output directory exists
os.makedirs("work/outputs", exist_ok=True)

# Columns to export
export_columns = [
    "rank",
    "decline_score",
    "recommended_action",
    "reason_code",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "search_volume",
    "competition",
    "word_count",
    "char_count",
    "content_type",
    "main_intent"
]

# Export ranked queue
queue_export = queue[export_columns].copy()

output_path = "work/outputs/ml10_ranked_action_queue.csv"

queue_export.to_csv(
    output_path,
    index=False
)

print("Export complete:")
print(output_path)

print("\nExport shape:", queue_export.shape)

print("\nFirst 10 rows:")
display(queue_export.head(10))

Export complete:
work/outputs/ml10_ranked_action_queue.csv

Export shape: (7115, 16)

First 10 rows:


,rank,decline_score,recommended_action,reason_code,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,days_since_last_update,content_age_days,avg_position,search_volume,competition,word_count,char_count,content_type,main_intent
0,1,0.864996,REVIEW_REFRESH,HIGH_MODEL_SCORE,84773,19,19,20,95,7.8,0.0,0.00,2871.0,19536.0,keyword article,informational
1,2,0.839192,REVIEW_REFRESH,HIGH_MODEL_SCORE,64917,6,11,20,97,7.2,170.0,0.02,2810.0,19393.0,keyword article,informational
2,3,0.838248,REVIEW_REFRESH,HIGH_MODEL_SCORE,97200,84,44,20,97,4.7,10.0,0.00,2658.0,17814.0,keyword article,informational
3,4,0.806764,REVIEW_REFRESH,HIGH_MODEL_SCORE,97606,30,45,20,280,2.3,10.0,1.00,3528.0,24856.0,keyword article,informational
4,5,0.803101,REVIEW_REFRESH,VISIBLE_STALE,30774,27,15,106,106,8.7,0.0,0.00,2691.0,17851.0,keyword article,informational
5,6,0.798786,REVIEW_REFRESH,HIGH_MODEL_SCORE,54506,13,17,20,95,21.8,0.0,0.00,2941.0,20081.0,keyword article,informational
6,7,0.777023,REVIEW_REFRESH,HIGH_MODEL_SCORE,84550,114,57,20,97,5.6,14800.0,0.00,2869.0,19949.0,keyword article,informational
7,8,0.775210,REVIEW_REFRESH,VISIBLE_STALE,9919,8,13,106,106,4.6,0.0,0.00,2606.0,17227.0,keyword article,informational
8,9,0.775169,REVIEW_REFRESH,HIGH_MODEL_SCORE,21552,15,21,89,105,6.7,10.0,0.04,2490.0,17573.0,keyword article,informational
9,10,0.774384,REVIEW_REFRESH,VISIBLE_STALE,218786,250,181,104,537,4.2,1900.0,0.00,NaN,NaN,keyword article,informational


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.